In [1]:
import json
from typing import Optional, Sequence, Tuple, Union, List

import numpy as np

from pyrtid.forward import ForwardModel, Geometry
from pyrtid.utils.enum import StrEnum
from pyrtid.utils.types import (
    NDArrayFloat,
    NDArrayInt,
    object_or_object_sequence_to_list,
)

from pyrtid.utils import get_mean_values_for_last_axis, MeanType, node_number_to_indices
from pyrtid.utils import finite_gradient

from pyrtid.inverse.obs import (
    Observable,
    StateVariable,
    get_times_idx_before_after_obs,
    get_weights,
    get_values_matching_node_indices,
)

In [3]:
def get_simu_values(
    obs_times: NDArrayFloat,
    calc_times: NDArrayFloat,
    obs_values: NDArrayFloat,
    calc_values: NDArrayFloat,
) -> NDArrayFloat:
    """
    Get an array of residuals between the observed values and the calculated values.

    Note
    ----
    When an observed values does not match calculated values, then an equivalent
    calculated value is computed (TODO: add ref to paper).

    Parameters
    ----------
    obs_times : NDArrayFloat
        Array of observation times.
    calc_times : NDArrayFloat
        Array of calculated times.
    obs_values : NDArrayFloat
        Array of observed values.
    calc_values : NDArrayFloat
        Array of calculated values.

    Returns
    -------
    NDArrayFloat
        Residuals between simulated values and observed values.

    """
    idx_before, idx_after = get_times_idx_before_after_obs(obs_times, calc_times)
    weights_before, weights_after = get_weights(
        obs_times, calc_times, idx_before, idx_after
    )
    return (
        weights_before * calc_values[idx_before]
        + weights_after * calc_values[idx_after]
    )


def get_residuals(
    obs_times: NDArrayFloat,
    calc_times: NDArrayFloat,
    obs_values: NDArrayFloat,
    calc_values: NDArrayFloat,
) -> NDArrayFloat:
    """
    Get an array of residuals between the observed values and the calculated values.

    Note
    ----
    When an observed values does not match calculated values, then an equivalent
    calculated value is computed (TODO: add ref to paper).

    Parameters
    ----------
    obs_times : NDArrayFloat
        Array of observation times.
    calc_times : NDArrayFloat
        Array of calculated times.
    obs_values : NDArrayFloat
        Array of observed values.
    calc_values : NDArrayFloat
        Array of calculated values.

    Returns
    -------
    NDArrayFloat
        Residuals between simulated values and observed values.

    """
    idx_before, idx_after = get_times_idx_before_after_obs(obs_times, calc_times)
    weights_before, weights_after = get_weights(
        obs_times, calc_times, idx_before, idx_after
    )
    return obs_values - (
        weights_before * calc_values[idx_before]
        + weights_after * calc_values[idx_after]
    )


def least_squares(
    obs: Observable,
    calc_values: NDArrayFloat,
    ldt: List[float],
    mean_type: MeanType,
    hm_end_time: Optional[float] = None,
) -> float:
    """
    Get the least square (L2) cost function value for a given obervable.

    .. math::
        \mathcal{J} = \dfrac{1}{2\lvert \bm{d}_{\mathrm{obs}} \rvert} \sum_{n=0}^{N}
        \left(\dfrac{d_{\mathrm{obs}}^{n}
        - d_{\mathrm{calc}}^{n}}{\sigma_{\mathrm{obs}}^{n}} \right)^{2}

    with $n$, a time with an observation, and $\lvert \bm{d}_{\mathrm{obs}} \rvert$
    the number of observation points.

    Parameters
    ----------
    obs : Observable
        Observable instance.
    calc_values : NDArrayFloat
        Array of calculated values with shape (nx, ny, nt)
    ldt : List[float]
        List of timesteps between calculated values with length nt - 1.
    interpolation_type: InterpolationType
        Type of interpolation chosen for the simulated value when the observed one
        is defined over several meshes of the domain.
    hm_end_time: Optional[float]
        End of history matching. After this time, observations are ignored.
        The default is None.

    Returns
    -------
    float
        _description_
    """
    # times of simulated values
    simu_times = np.cumsum([0] + ldt)
    obs_times, obs_values, obs_uncertainties = get_obs_times_vals_stds(
        obs, simu_times, hm_end_time
    )
    calc_values = get_mean_values_for_last_axis(
        get_values_matching_node_indices(obs.node_indices, calc_values),
        mean_type=mean_type,
    )
    res = get_residuals(obs_times, simu_times, obs_values, calc_values)
    return 0.5 * np.sum(np.square(res / obs_uncertainties))


def least_squares_derivative(
    obs: Observable,
    calc_values: NDArrayFloat,
    ldt: List[float],
    geometry: Geometry,
    mean_type: MeanType,
) -> NDArrayFloat:
    r"""

    .. math::
        \dfrac{\partial\mathcal{J}}{\partial d_{\mathrm{obs}}^{n}} =
        \dfrac{\partial}{\partial d_{\mathrm{obs}}^{n}} \left(\dfrac{1}{2 \lvert
        \bm{d}_{\mathrm{obs}} \rvert } \sum_{n=0}^{N} \left(\dfrac{d_{\mathrm{obs}}^{n}
        - d_{\mathrm{calc}}^{n}}{\sigma_{\mathrm{obs}}^{n}} \right)^{2}\right) =
        - \dfrac{1}{\lvert \bm{d}_{\mathrm{obs}} \rvert } \dfrac{d_{\mathrm{obs}}^{n}
        - d_{\mathrm{calc}}^{n}}{\left(\sigma_{\mathrm{obs}}^{n} \right)^{2}}$$

    with $n$, a time with an observation, and $\lvert \bm{d}_{\mathrm{obs}} \rvert$
    the number of observation points.

    Parameters
    ----------
    obs : Observable
        Observable instance.
    calc_values : NDArrayFloat
        Array of calculated values with shape (nx, ny, nt)
    ldt : List[float]
        List of timesteps between calculated values with length nt - 1.
    geometry : Geometry
        Geometry of the problem.

    Returns
    -------
    NDArrayFloat
        _description_
    """
    simu_times = np.cumsum([0] + ldt)
    obs_times, obs_values, obs_uncertainties = get_obs_times_vals_stds(
        obs, simu_times, hm_end_time
    )
    idx_before, idx_after = get_times_idx_before_after_obs(obs_times, simu_times)
    weights_before, weights_after = get_weights(
        obs_times, simu_times, idx_before, idx_after
    )

    derivative = np.zeros((geometry.nx, geometry.ny, simu_times.size))

    res = get_residuals(obs_times, simu_times, obs_values, calc_values)

    derivative[*obs.node_indices, idx_before] = (
        -res * weights_before / obs.uncertainties**2
    )
    derivative[*obs.node_indices, idx_after] = (
        -res * weights_after / obs.uncertainties**2
    )

    return -1 * derivative

$$ \mathcal{J} = \dfrac{1}{2 \lvert \bm{d}_{\mathrm{obs}} \rvert } \sum_{n=0}^{N} \left(\dfrac{d_{\mathrm{obs}}^{n} - d_{\mathrm{calc}}^{n}}{\sigma_{\mathrm{obs}}^{n}} \right)^{2}$$

$$ \dfrac{\partial\mathcal{J}}{\partial d_{\mathrm{obs}}^{n}} = \dfrac{\partial}{\partial d_{\mathrm{obs}}^{n}} \left(\dfrac{1}{2 \lvert \bm{d}_{\mathrm{obs}} \rvert } \sum_{n=0}^{N} \left(\dfrac{d_{\mathrm{obs}}^{n} - d_{\mathrm{calc}}^{n}}{\sigma_{\mathrm{obs}}^{n}} \right)^{2}\right) = - \dfrac{1}{\lvert \bm{d}_{\mathrm{obs}} \rvert } \dfrac{d_{\mathrm{obs}}^{n} - d_{\mathrm{calc}}^{n}}{\left(\sigma_{\mathrm{obs}}^{n} \right)^{2}}$$

- Need to make sure that observation after the simulation end are not taken into account

In [19]:
obs = Observable(
    StateVariable.CONCENTRATION,
    node_indices=[2, 4, 6],
    times=np.array([0.0, 2.0, 4.0, 5.6]),
    values=np.array([1, 1, 1, 1]),
    uncertainties=np.array([0.5, 0.5, 0.5, 0.5]),
)

geometry = Geometry(3, 3, 1.0, 1.0, 1.0)

ldt = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
calc_values = np.ones((3, 3, 10))

obs_times = obs.times

In [20]:
least_squares(obs, calc_values, ldt, mean_type=MeanType.ARITHMETIC)
least_squares(obs, calc_values, ldt, mean_type=MeanType.GEOMETRIC)
least_squares(obs, calc_values, ldt, mean_type=MeanType.HARMONIC)

0.0

In [21]:
calc_values = np.ones((3, 3, 10)) * 2.0

least_squares(obs, calc_values, ldt, mean_type=MeanType.ARITHMETIC)
least_squares(obs, calc_values, ldt, mean_type=MeanType.GEOMETRIC)
least_squares(obs, calc_values, ldt, mean_type=MeanType.HARMONIC)

2.0

In [ ]:
least_squares_derivative(obs, calc_values, ldt, geometry)

array([[[ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072],
        [ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072],
        [ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072]],

       [[ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072],
        [ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072],
        [ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072]],

       [[ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072],
        [ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072],
        [ 1.   ,  0.   ,  0.6  ,  0.   ,  0.2  , -0.048, -0.072]]])

In [ ]:
obs_times = obs.times[obs_times < calc_times[-1]]